# Day 3 â€” Hands-On Lab 1: Public API + GraphDB Exploration
### GlobalMart Data Engineering Bootcamp

| | |
|---|---|
| **Follows** | ILT 1 â€” API Ingestion Mechanics + Intro to GraphDB/Cypher Basics |
| **Time** | 11:00 AM â€“ 1:00 PM (2 hours) |
| **Output** | A sandbox Delta table of FX rates, a small graph in Neo4j AuraDB, and a sandbox Delta table read back from that graph |

> **Side-exploration, not part of the GlobalMart build.** Exactly like ILT 1 said: GlobalMart's real pipeline only has two sources â€” Postgres CDC (Lakeflow Connect) and ADLS Autoloader. Nothing in this lab feeds Bronze/Silver/Gold or `fact_sales`. You're practicing two patterns (REST API ingestion, graph databases) that you will meet on *other* projects â€” GlobalMart-shaped data is just a familiar example.

### What you will build
**Part 1 â€” REST API (45 min):** Extend the frankfurter.app demo from ILT 1 â€” pull multiple base currencies, then pull historical rates for more than one date, and land both as sandbox Delta tables.

**Part 2 â€” GraphDB (75 min):** Create a free Neo4j AuraDB instance, seed it with a small GlobalMart-shaped graph (Customer â†’ Order â†’ Product â†’ Supplier) using Cypher, query it from the Neo4j Browser, then connect to it from Databricks and land a query result as a sandbox Delta table.

**Instructions:** Run each cell with **Shift + Enter**. Cells marked `### YOUR TURN` are for you to complete â€” the pattern is always demonstrated once above them first.

---
## Part 1 â€” REST API: Beyond a Single Call

ILT 1 called `https://api.frankfurter.app/latest` once, for the default base currency (EUR), and saved the result to `sandbox_path/fx_rates`. Two realistic extensions:

1. **Multiple base currencies** â€” "What if GlobalMart sells in USD, GBP, *and* INR?" You need one row set per base currency, not just one.
2. **Historical rates** â€” "What was the rate last week?" frankfurter.app supports a date in the URL path: `https://api.frankfurter.app/2026-01-01` returns rates as of that date.

In [ ]:
# â”€â”€â”€ Setup: same Unity Catalog External Location as ILT 1 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# No storage key needed â€” this cluster already has access via the External
# Location's Managed Identity (set up Day 2 HOL 1).

EXTERNAL_LOCATION = "abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net"
sandbox_path = f"{EXTERNAL_LOCATION}/sandbox/api_graphdb"
print(f"Sandbox path: {sandbox_path}")

In [ ]:
# â”€â”€â”€ Step 1: Multiple base currencies â€” one API call per currency â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# frankfurter.app takes a `base` query parameter. We loop over the currencies
# GlobalMart actually cares about and combine every response into one table.

import requests
from pyspark.sql import Row
from datetime import datetime

base_currencies = ["USD", "EUR", "GBP"]  # GlobalMart's three storefront currencies
all_rows = []

for base in base_currencies:
    response = requests.get("https://api.frankfurter.app/latest", params={"base": base})
    print(f"Called base={base} -> status {response.status_code}")

    if response.status_code != 200:
        # Don't silently skip a failed call â€” a DE pipeline should always know
        # when a source didn't return what was expected.
        raise RuntimeError(f"API call failed for base={base}: {response.status_code}")

    data = response.json()
    for currency, rate in data["rates"].items():
        all_rows.append(Row(
            base_currency   = data["base"],
            target_currency = currency,
            exchange_rate   = float(rate),
            rate_date       = data["date"],
            ingested_at     = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        ))

multi_base_df = spark.createDataFrame(all_rows)
print(f"\nTotal rows across {len(base_currencies)} base currencies: {multi_base_df.count()}")
multi_base_df.filter(multi_base_df.target_currency.isin("USD", "EUR", "GBP", "INR")).show()

In [ ]:
# â”€â”€â”€ Step 2: Save the multi-currency table to sandbox â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# overwrite is correct here â€” like ILT 1 explained, this is a live snapshot,
# not something we want to accumulate duplicate-but-stale rows for.

multi_base_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{sandbox_path}/fx_rates_multi_base")

saved = spark.read.format("delta").load(f"{sandbox_path}/fx_rates_multi_base")
print(f"Saved {saved.count()} rows to {sandbox_path}/fx_rates_multi_base")
saved.groupBy("base_currency").count().show()

### YOUR TURN â€” Historical Rates

frankfurter.app accepts a date in place of `latest`: `https://api.frankfurter.app/2026-01-01`.

Complete the cell below to:
1. Loop over the two dates given in `historical_dates`
2. Call the API for each date (base currency EUR is fine â€” keep it simple)
3. Build one combined DataFrame with a `rate_date` column that reflects the **requested** date (not `data["date"]`, in case the API adjusts to the nearest business day â€” compare the two and see!)
4. Append (not overwrite!) into `sandbox_path/fx_rates_historical` â€” this is the "accumulate history" case ILT 1 mentioned

In [ ]:
# â”€â”€â”€ YOUR TURN: fill in the two TODOs below â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
historical_dates = ["2026-01-01", "2026-02-01"]
historical_rows  = []

for requested_date in historical_dates:
    # TODO 1: call the API for this date instead of "latest"
    # Hint: api_url = f"https://api.frankfurter.app/{requested_date}"
    api_url = None  # â† replace this line

    response = requests.get(api_url)
    print(f"Requested {requested_date} -> API returned date {response.json().get('date') if response.status_code == 200 else 'ERROR'}")

    if response.status_code != 200:
        raise RuntimeError(f"API call failed for {requested_date}: {response.status_code}")

    data = response.json()
    for currency, rate in data["rates"].items():
        historical_rows.append(Row(
            base_currency    = data["base"],
            target_currency  = currency,
            exchange_rate    = float(rate),
            requested_date   = requested_date,      # what we asked for
            api_returned_date = data["date"],        # what we actually got (may differ on weekends/holidays)
            ingested_at      = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        ))

historical_df = spark.createDataFrame(historical_rows)

# TODO 2: write this as APPEND, not overwrite â€” we're building history over time
historical_df.write \
    .format("delta") \
    .mode("PUT_THE_RIGHT_MODE_HERE") \
    .save(f"{sandbox_path}/fx_rates_historical")

print(f"Rows in historical table now: {spark.read.format('delta').load(f'{sandbox_path}/fx_rates_historical').count()}")

---
## Part 2 â€” GraphDB: Build and Query a Small GlobalMart Graph

### Step 1 â€” Create your free Neo4j AuraDB instance

1. Go to **`neo4j.com/cloud/aura`** and sign up for a free account.
2. Click **Create Free Instance** (AuraDB Free tier â€” no credit card required).
3. Give it any name, e.g. `globalmart-graph-demo`.
4. When it's created, Neo4j shows you a **connection URI**, **username** (always `neo4j`), and a **generated password** â€” download or copy these now. The password is shown **only once**.
5. Wait ~1â€“2 minutes for the instance status to turn from "Creating" to "Running".
6. Click **Open** to launch the **Neo4j Browser** â€” a web UI where you'll paste Cypher directly (Steps 2â€“3 below happen here, not in Databricks).

> Keep the Neo4j Browser tab open â€” you'll go back and forth between it and this notebook.

### Step 2 â€” Seed the graph (paste into Neo4j Browser, not here)

This models a tiny slice of GlobalMart as a graph: 3 customers, 3 products, 2 suppliers, 3 orders, wired together with the same relationships ILT 1 introduced (`PLACED`, `CONTAINS`, `SUPPLIED_BY`).

Paste this whole block into the Neo4j Browser query bar and run it (â–¶ or Ctrl+Enter):

```cypher
// Nodes
CREATE (c1:Customer {customer_id: 'CUST-001', name: 'Raj Patel',   city: 'Mumbai'})
CREATE (c2:Customer {customer_id: 'CUST-002', name: 'Priya Singh', city: 'Bangalore'})
CREATE (c3:Customer {customer_id: 'CUST-003', name: 'Arjun Mehta', city: 'Delhi'})

CREATE (p1:Product {product_id: 'PRD-001', name: 'Wireless Mouse', category: 'Electronics'})
CREATE (p2:Product {product_id: 'PRD-002', name: 'Office Chair',   category: 'Furniture'})
CREATE (p3:Product {product_id: 'PRD-003', name: 'Desk Lamp',      category: 'Furniture'})

CREATE (s1:Supplier {supplier_id: 'SUP-001', name: 'TechDistributors Inc'})
CREATE (s2:Supplier {supplier_id: 'SUP-002', name: 'HomeOffice Supplies'})

CREATE (o1:Order {order_id: 'ORD-001', order_date: '2026-06-01'})
CREATE (o2:Order {order_id: 'ORD-002', order_date: '2026-06-03'})
CREATE (o3:Order {order_id: 'ORD-003', order_date: '2026-06-05'})

// Relationships â€” Customer -[PLACED]-> Order
WITH 1 AS dummy
MATCH (c:Customer {customer_id: 'CUST-001'}), (o:Order {order_id: 'ORD-001'}) CREATE (c)-[:PLACED]->(o)
WITH 1 AS dummy
MATCH (c:Customer {customer_id: 'CUST-002'}), (o:Order {order_id: 'ORD-002'}) CREATE (c)-[:PLACED]->(o)
WITH 1 AS dummy
MATCH (c:Customer {customer_id: 'CUST-001'}), (o:Order {order_id: 'ORD-003'}) CREATE (c)-[:PLACED]->(o)

// Relationships â€” Order -[CONTAINS]-> Product
WITH 1 AS dummy
MATCH (o:Order {order_id: 'ORD-001'}), (p:Product {product_id: 'PRD-001'}) CREATE (o)-[:CONTAINS]->(p)
WITH 1 AS dummy
MATCH (o:Order {order_id: 'ORD-002'}), (p:Product {product_id: 'PRD-002'}) CREATE (o)-[:CONTAINS]->(p)
WITH 1 AS dummy
MATCH (o:Order {order_id: 'ORD-003'}), (p:Product {product_id: 'PRD-003'}) CREATE (o)-[:CONTAINS]->(p)

// Relationships â€” Product -[SUPPLIED_BY]-> Supplier
WITH 1 AS dummy
MATCH (p:Product {product_id: 'PRD-001'}), (s:Supplier {supplier_id: 'SUP-001'}) CREATE (p)-[:SUPPLIED_BY]->(s)
WITH 1 AS dummy
MATCH (p:Product {product_id: 'PRD-002'}), (s:Supplier {supplier_id: 'SUP-002'}) CREATE (p)-[:SUPPLIED_BY]->(s)
WITH 1 AS dummy
MATCH (p:Product {product_id: 'PRD-003'}), (s:Supplier {supplier_id: 'SUP-002'}) CREATE (p)-[:SUPPLIED_BY]->(s)
```

Run `MATCH (n) RETURN n` afterwards in the Browser to see the graph visualization â€” 11 nodes, 9 relationships.

### Step 3 â€” Try Cypher queries in the Neo4j Browser

Paste each of these one at a time and look at the result table (or switch to the graph view):

```cypher
// Which products did each customer buy?
MATCH (c:Customer)-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
RETURN c.name AS customer_name, p.name AS product_name, o.order_id AS order_id
```

### YOUR TURN (in the Neo4j Browser)
Write and run a Cypher query that counts how many orders each customer placed, sorted highest first. (Same shape as ILT 1's pattern #4 â€” `MATCH ... RETURN ... COUNT(o) ... ORDER BY`.)

---
## Part 3 â€” Read the Graph Into Databricks

Back in this notebook now. We connect to AuraDB from Databricks using the official `neo4j` Python driver, run the same customerâ†’product query as Step 3, and land the result as a sandbox Delta table â€” exactly the same shape of task as the API section, just a different source.

In [ ]:
# â”€â”€â”€ Install the Neo4j Python driver on this cluster â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
%pip install neo4j

In [ ]:
# â”€â”€â”€ Connect to your AuraDB instance â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Paste the 3 values Neo4j showed you when the instance was created in Step 1.
# NOTE: never commit real credentials â€” replace these placeholders locally,
# then swap them back to placeholders before you upload/submit this notebook.

from neo4j import GraphDatabase

NEO4J_URI      = "neo4j+s://YOUR_INSTANCE_ID.databases.neo4j.io"  # â† replace
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "YOUR_AURADB_PASSWORD"  # â† replace

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, params=None):
    """Run one Cypher query against AuraDB and return a list of plain dicts."""
    with driver.session() as session:
        result = session.run(query, params or {})
        return [dict(record) for record in result]

# Quick connectivity check â€” should print 11 (3 customers + 3 products + 2 suppliers + 3 orders)
node_count = run_cypher("MATCH (n) RETURN count(n) AS total")[0]["total"]
print(f"Connected! Total nodes in the graph: {node_count}")

In [ ]:
# â”€â”€â”€ Run the same customer -> product query from Step 3, but from Databricks â”€â”€â”€

customer_product_query = """
MATCH (c:Customer)-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
RETURN c.customer_id AS customer_id, c.name AS customer_name,
       p.product_id  AS product_id,  p.name AS product_name,
       o.order_id    AS order_id,    o.order_date AS order_date
"""

rows = run_cypher(customer_product_query)
print(f"Rows returned from Neo4j: {len(rows)}")
for r in rows:
    print(r)

In [ ]:
# â”€â”€â”€ Convert to a Spark DataFrame and save to sandbox â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Same pattern as the API section: land in sandbox/, never bronze/ â€” this is
# exploration output, not a GlobalMart pipeline table.

graph_df = spark.createDataFrame(rows)
graph_df.show(truncate=False)

graph_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{sandbox_path}/graph_customer_orders")

print(f"Saved to: {sandbox_path}/graph_customer_orders")

### YOUR TURN â€” Supplier Traversal

Write a Cypher query (as a Python string, same shape as `customer_product_query` above) that returns, for every product, which supplier it comes from â€” a 2-hop traversal (`Product -[:SUPPLIED_BY]-> Supplier`) plus which orders contained that product (`Order -[:CONTAINS]-> Product`). Run it through `run_cypher(...)`, convert to a DataFrame, and save it to `sandbox_path/graph_product_suppliers`.

In [ ]:
# â”€â”€â”€ YOUR TURN: complete this query and save the result â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

product_supplier_query = """
# TODO: MATCH a path across Order -[:CONTAINS]-> Product -[:SUPPLIED_BY]-> Supplier
# RETURN order_id, product_name, supplier_name
"""

# Uncomment once the query above is complete:
# supplier_rows = run_cypher(product_supplier_query)
# supplier_df = spark.createDataFrame(supplier_rows)
# supplier_df.show(truncate=False)
# supplier_df.write.format("delta").mode("overwrite").save(f"{sandbox_path}/graph_product_suppliers")

In [ ]:
# â”€â”€â”€ Close the Neo4j driver connection when done â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Good hygiene â€” an open driver holds a connection pool open on AuraDB's side.

driver.close()
print("Neo4j driver closed.")

---
## Submission Checklist

```
Submission Checklist
â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
âœ… Multi-base-currency FX table saved  â†’ sandbox/fx_rates_multi_base
âœ… Historical FX table saved (append mode) â†’ sandbox/fx_rates_historical
â”€â”€ Rows in historical table:          ______
âœ… Neo4j AuraDB Free instance created and graph seeded (11 nodes, 9 relationships)
âœ… Customer -> Order -> Product query run in Neo4j Browser
âœ… Same query run from Databricks via the neo4j Python driver
âœ… Result saved â†’ sandbox/graph_customer_orders
âœ… Supplier traversal exercise completed â†’ sandbox/graph_product_suppliers
âœ… Before submitting: replaced NEO4J_URI/PASSWORD with placeholders again
â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
```

**Reminder:** none of this feeds `fact_sales`. You practiced two ingestion *patterns* â€” REST API and graph traversal â€” that show up in other projects, using GlobalMart as a familiar backdrop.

---
## Next â€” Day 3 ILT 2 (2:00 PM â€“ 3:00 PM)
**Autoloader & Schema Evolution Concepts**